In [ ]:
# python + mysql => pymysql
# python + mongodb => pymongo
# mongoDB -> Binary + json = BSON => 객체의 형태를 띄고 있음

In [1]:
!pip install pymongo

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [10]:
import requests
from bs4 import BeautifulSoup
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["ecommerce"]
collection = db["teddyproducts"]

documents = []

for page_num in range(8) :
    if page_num == 0 :
        res = requests.get("https://davelee-fun.github.io/")
    else :
        requests.get(f"https://davelee-fun.github.io/page{page_num + 1}")
    
    soup = BeautifulSoup(res.content, "html.parser")
    data = soup.select("div.card-body")

    for item in data :
        category = item.select_one("h2.card-title").get_text().replace("관련 상품 추천", "").strip()
        product = item.select_one("h4.card-text").get_text().replace("상품명:", "").strip()

        doc = {
            "title": product,
            "category": category
        }
        
        documents.append(doc)

if documents :
    collection.insert_many(documents)

client.close()